In [2]:
import sys
import os
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import models
sys.path.append("../src") 
from data.dataset import RAFCETrainDataset, RAFCEValDataset, get_weighted_sampler
from models.train_utils import train_model

In [3]:
train_df = pd.read_csv("../data/train_split.csv")
val_df = pd.read_csv("../data/val_split.csv")

In [4]:
train_ds = RAFCETrainDataset(train_df, img_dir="../data/train_images")
val_ds = RAFCEValDataset(val_df, img_dir="../data/val_images")
sampler = get_weighted_sampler(train_df)
train_loader = DataLoader(train_ds, batch_size=32, sampler=sampler, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=0)

Our goal is to check: "How accurately can a standard model predict these specific 14 emotions?" To measure this accuracy, the model typically needs to output a probability for each of the 14 classes.

Input: Image
ResNet Body: Extracts deep features (e.g., a vector of size 2048).
FC Layer (nn.Linear(..., 14)): Maps those 2048 features to 14 scores compared to the ground truth labels.

# ResNet

In [5]:
model_ft = models.resnet50(pretrained=True)
model_ft.fc = nn.Linear(model_ft.fc.in_features, 14)

c:\Users\hedil\anaconda3\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\hedil\anaconda3\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to C:\Users\hedil/.cache\torch\hub\checkpoints\resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [07:42<00:00, 222kB/s] 


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model_ft.parameters(), lr=1e-4)
model, history = train_model(
    model_ft, train_loader, val_loader, 
    criterion, optimizer, 
    num_epochs=15, device=device
)

Epoch 1/15
----------
train Loss: 1.6038 Acc: 0.4707 F1-Macro: 0.4639
val Loss: 2.0813 Acc: 0.3640 F1-Macro: 0.1594
--> New Best Model Saved (F1: 0.1594)

Epoch 2/15
----------
train Loss: 0.8554 Acc: 0.7145 F1-Macro: 0.7081
val Loss: 2.0786 Acc: 0.4180 F1-Macro: 0.1893
--> New Best Model Saved (F1: 0.1893)

Epoch 3/15
----------
train Loss: 0.5998 Acc: 0.7968 F1-Macro: 0.7973
val Loss: 2.0975 Acc: 0.4494 F1-Macro: 0.1769

Epoch 4/15
----------
train Loss: 0.4009 Acc: 0.8685 F1-Macro: 0.8697
val Loss: 1.8954 Acc: 0.4730 F1-Macro: 0.2039
--> New Best Model Saved (F1: 0.2039)

Epoch 5/15
----------
train Loss: 0.2758 Acc: 0.9189 F1-Macro: 0.9196
val Loss: 2.2348 Acc: 0.4472 F1-Macro: 0.1802

Epoch 6/15
----------
train Loss: 0.2306 Acc: 0.9244 F1-Macro: 0.9273
val Loss: 2.1840 Acc: 0.4865 F1-Macro: 0.2285
--> New Best Model Saved (F1: 0.2285)

Epoch 7/15
----------
train Loss: 0.1889 Acc: 0.9413 F1-Macro: 0.9401
val Loss: 2.1353 Acc: 0.4584 F1-Macro: 0.2027

Epoch 8/15
----------
train L